In [ ]:
import re
import pymupdf
import pandas as pd

import unicodedata
from pathlib import Path

import json
from typing import Literal

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from pydantic import BaseModel

import time

In [ ]:
COLS = ["data", "merchant", "valor_pago", "categoria", "pagamento", "revisar"]
PARSERS = {}

def parser(nome, detecta):
    def deco(fn):
        PARSERS[nome] = (detecta, fn)
        return fn
    return deco

def pdf_text(path) -> str:
    with pymupdf.open(path) as doc:
        return "\n".join(p.get_text("text", sort=True) for p in doc)

def to_float(s: str) -> float:
    return float(s.replace(".", "").replace(",", "."))

VENCIMENTO = re.compile(r"Vencimento\s+[^\n]*?R\$\s*[\d.]+,\d{2}\s+\d{2}/(\d{2})/(\d{4})")
LINHA_FATURA = re.compile(r"^[ \t]*(\d{2}/\d{2})[ \t]+(\S.*)$", re.M)
VALOR = re.compile(r"(?<![\d,.])(\d{1,3}(?:\.\d{3})*,\d{2})(?![\d%])([ \t]*-)?")
COTACAO = re.compile(r"\d,\d{4}[ \t]+(\d{1,3}(?:\.\d{3})*,\d{2})")
PARCELA = re.compile(r"^\d{2}/\d{2}$")

def eh_fatura_bradesco(texto: str) -> bool:
    t = texto.lower()
    return "fatura_bradesco" in t or ("bradesco" in t and "vencimento" in t)

@parser("fatura_bradesco", eh_fatura_bradesco)
def parse_fatura_bradesco(texto: str) -> pd.DataFrame:
    mes_venc, ano_venc = map(int, VENCIMENTO.search(texto).groups())
    rows = []
    for dm, resto in LINHA_FATURA.findall(texto):
        valores = list(VALOR.finditer(resto))
        if not valores:
            continue
        # compra internacional: "USD 5,00 ... 5,00 5,4400 27,20" → o valor em R$ vem logo após a cotação
        cotacao = COTACAO.search(resto)
        valor = to_float(cotacao.group(1) if cotacao else valores[0].group(1))
        if not cotacao and valores[0].group(2):  # "9.878,97 -" é crédito
            valor = -valor
        # o texto antes do valor vem em colunas separadas por 2+ espaços: merchant, parcela (opcional), cidade
        campos = re.split(r"\s{2,}", resto[: valores[0].start()].strip())
        merchant = campos[0]
        if len(campos) > 1 and PARCELA.match(campos[1]):
            merchant += f" {campos[1]}"
        dia, mes = map(int, dm.split("/"))
        # compra de dezembro numa fatura que vence em janeiro → ano anterior
        ano = ano_venc - 1 if mes > mes_venc else ano_venc
        rows.append((pd.Timestamp(ano, mes, dia), merchant, valor))

    df = pd.DataFrame(rows, columns=["data", "merchant", "valor_pago"])
    df = df[~df.merchant.str.contains(r"PAGTO\. POR DEB EM C/C|Pagamento de fatura", case=False)]
    df["pagamento"] = "Bradesco"
    return df.reset_index(drop=True)

def ler_fatura(path) -> pd.DataFrame:
    """Extrai o texto do PDF e usa o primeiro parser registrado que reconhecer o formato."""
    texto = pdf_text(path)
    for nome, (detecta, fn) in PARSERS.items():
        if detecta(texto):
            print(f"{Path(path).name}: lido com o parser '{nome}'.")
            return fn(texto)
    raise ValueError(f"Nenhum parser reconheceu {Path(path).name}. Parsers registrados: {list(PARSERS)}")

In [ ]:
REFERENCIA = Path("/Users/eduar/o-que-contas/reference/Consolidado__Despesas_Distintos.csv")

def normaliza(nome: str) -> str:
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode().upper()
    nome = re.sub(r"\b\d{2}/\d{2}\b", " ", nome)  # parcela: "01/12"
    nome = re.sub(r"-\d+\b", " ", nome)           # loja: "-0001"
    nome = re.sub(r"[^A-Z0-9 ]", " ", nome)       # "*", ".", "/"
    return re.sub(r"\s+", " ", nome).strip()

def validar_referencia(ref: pd.DataFrame) -> pd.Series:
    """Retorna as categorias possíveis dos merchants ambíguos (mesmo nome normalizado, categorias diferentes)."""
    vazios = ref[["merchant", "categoria"]].isna().any(axis=1)
    if vazios.any():
        raise ValueError(f"Referência com merchant/categoria vazio nas linhas {list(ref.index[vazios] + 2)}")

    categorias = ref.groupby("norm").categoria.unique()
    ambiguos = categorias[categorias.map(len) > 1].map(sorted)
    for norm, opcoes in ambiguos.items():
        print(f"Ambíguo: '{norm}' aparece como {opcoes}; essas linhas vão para revisão.")

    variantes = ref.groupby("norm").merchant.unique()
    variantes = variantes[(variantes.map(len) > 1) & ~variantes.index.isin(ambiguos.index)]
    if len(variantes):
        print(f"{len(variantes)} merchant(s) com grafias redundantes (mesma categoria), ex.: {list(variantes.iloc[0])}")
    return ambiguos

def carregar_referencia(path=REFERENCIA) -> tuple[pd.Series, pd.Series]:
    ref = pd.read_csv(path)
    ref["norm"] = ref.merchant.map(normaliza)
    ambiguos = validar_referencia(ref)
    referencia = ref[~ref.norm.isin(ambiguos.index)].drop_duplicates("norm").set_index("norm").categoria
    return referencia, ambiguos

def categorizar(df: pd.DataFrame, referencia: pd.Series, ambiguos: pd.Series) -> pd.DataFrame:
    df = df.copy()
    norm = df.merchant.map(normaliza)
    df["categoria"] = norm.map(referencia)
    df["revisar"] = df.categoria.isna()
    df["opcoes"] = norm.map(ambiguos)  # categorias possíveis, só para merchants ambíguos
    return df

In [8]:
PATH = "/Users/eduar/o-que-contas/invoices_for_test/fatura_bradesco-8.pdf"

referencia, ambiguos = carregar_referencia()
df = categorizar(ler_fatura(PATH), referencia, ambiguos)

print(f"{(~df.revisar).sum()}/{len(df)} linhas categorizadas pela referência; "
      f"{df[df.revisar].merchant.nunique()} merchants para revisar.")
df[COLS]

6 merchant(s) com grafias redundantes (mesma categoria), ex.: ['APPLE.COM/BILL', 'APPLE.COM/BILL.']
32/51 linhas categorizadas pela referência; 19 merchants para revisar.


,data,merchant,valor_pago,categoria,pagamento,revisar
0,2026-09-25,CUSTO TRANS. EXTERIOR-IOF,0.95,Taxas e encargos,Bradesco,False
1,2026-08-27,Distribusion Luxor,873.38,Ônibus,Bradesco,False
2,2026-08-27,PAO DE ACUCAR-0001,116.41,Mercado,Bradesco,False
3,2026-08-27,PAO DE ACUCAR-0001,30.70,Mercado,Bradesco,False
4,2026-08-27,FARMAIS,19.87,NaN,Bradesco,True
5,2026-08-28,GRINDHOUSE UM POR CE,88.00,NaN,Bradesco,True
6,2026-08-28,MINUTO PA 1456,15.58,NaN,Bradesco,True
7,2026-08-29,IFD*DE VERAS RESTAURANTE,68.99,Restaurante,Bradesco,False
8,2026-08-29,A2F SERVICOS DE ESTAC,20.00,Estacionamento,Bradesco,False
9,2026-08-29,PAO DE ACUCAR-0001,83.57,Mercado,Bradesco,False


In [ ]:
load_dotenv("/Users/eduar/o-que-contas/.env")
MODELO = "claude-opus-5-5"
claude = ChatAnthropic(
    model=MODELO,
    max_tokens=16000,
    reasoning_effort="low",
    betas=["server-side-fallback-2026-07-01"],
    model_kwargs={"fallbacks": "default"},  # se o modelo recusar, a API tenta outro modelo
)

def categorizar_com_claude(df: pd.DataFrame, path=REFERENCIA) -> pd.DataFrame:
    """Uma chamada ao Claude para os merchants que a referência não resolveu (revisar=True)."""
    ref = pd.read_csv(path)
    categorias = tuple(sorted(ref.categoria.unique()))

    class Classificacao(BaseModel):
        id: int
        categoria: Literal[categorias]

    class Resposta(BaseModel):
        classificacoes: list[Classificacao]

    pendentes = df[df.revisar].drop_duplicates("merchant")
    if pendentes.empty:
        return df
    itens = [
        {"id": i, "merchant": r.merchant, **({"opcoes": list(r.opcoes)} if isinstance(r.opcoes, list) else {})}
        for i, r in enumerate(pendentes.itertuples())
    ]
    prompt = (
        "Classifique cada merchant de uma fatura de cartão de crédito brasileira em uma das categorias permitidas.\n"
        "- Use os exemplos de referência para entender como eu uso cada categoria (ex.: 'Feira' x 'Mercado').\n"
        "- Se o item tiver 'opcoes', escolha uma delas.\n"
        "- Se não der para identificar o tipo de estabelecimento pelo nome, use 'Outros'.\n"
        "- Retorne exatamente uma classificação por id.\n\n"
        f"<referencia>\n{ref[['merchant', 'categoria']].to_csv(index=False)}</referencia>\n\n"
        f"<merchants>\n{json.dumps(itens, ensure_ascii=False, indent=1)}\n</merchants>"
    )
    resposta = claude.with_structured_output(Resposta, method="json_schema", include_raw=True).invoke(prompt)
    raw = resposta["raw"]
    if raw.response_metadata.get("stop_reason") == "refusal":
        raise RuntimeError(f"Claude recusou a classificação: {raw.response_metadata.get('stop_details')}")
    if resposta["parsing_error"]:
        raise ValueError(f"Resposta fora do formato esperado: {resposta['parsing_error']}")

    por_id = {c.id: c.categoria for c in resposta["parsed"].classificacoes}
    faltando = set(range(len(itens))) - por_id.keys()
    if faltando:
        raise ValueError(f"Claude não classificou: {[itens[i]['merchant'] for i in sorted(faltando)]}")

    df = df.copy()
    sugeridas = {itens[i]["merchant"]: categoria for i, categoria in por_id.items()}
    df.loc[df.revisar, "categoria"] = df.loc[df.revisar, "merchant"].map(sugeridas)
    u = raw.usage_metadata
    print(f"Claude classificou {len(itens)} merchants ({u['input_tokens']} tokens de entrada, {u['output_tokens']} de saída).")
    return df

In [ ]:
inicio = time.time()
df = categorizar_com_claude(df)
print(f"{time.time() - inicio:.1f}s")
df[COLS]

Claude classificou 19 merchants (5315 tokens de entrada, 359 de saída).
3.8s


,data,merchant,valor_pago,categoria,pagamento,revisar
0,2026-09-25,CUSTO TRANS. EXTERIOR-IOF,0.95,Taxas e encargos,Bradesco,False
1,2026-08-27,Distribusion Luxor,873.38,Ônibus,Bradesco,False
2,2026-08-27,PAO DE ACUCAR-0001,116.41,Mercado,Bradesco,False
3,2026-08-27,PAO DE ACUCAR-0001,30.70,Mercado,Bradesco,False
4,2026-08-27,FARMAIS,19.87,Farmácia,Bradesco,True
5,2026-08-28,GRINDHOUSE UM POR CE,88.00,Restaurante,Bradesco,True
6,2026-08-28,MINUTO PA 1456,15.58,Mercado,Bradesco,True
7,2026-08-29,IFD*DE VERAS RESTAURANTE,68.99,Restaurante,Bradesco,False
8,2026-08-29,A2F SERVICOS DE ESTAC,20.00,Estacionamento,Bradesco,False
9,2026-08-29,PAO DE ACUCAR-0001,83.57,Mercado,Bradesco,False
